# Validation du réseau électrique de Trey

Ce notebook contrôle le classeur réseau avant son import dans Pandapower. Il ne modifie jamais le fichier Excel.

## 1. Imports et localisation du classeur

La recherche ci-dessous fonctionne si le notebook est lancé depuis son dossier ou depuis la racine du dépôt.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

NOM_CLASSEUR = "2026_trey_power_network_completed.xlsx"
candidats = [
    Path(NOM_CLASSEUR),
    Path("RHT_Laboratory") / "GCG" / NOM_CLASSEUR,
]
chemin_reseau = next((p for p in candidats if p.exists()), None)
if chemin_reseau is None:
    raise FileNotFoundError(f"Classeur introuvable. Dossier courant : {Path.cwd()}")

chemin_reseau = chemin_reseau.resolve()
print("Classeur utilisé :", chemin_reseau)

## 2. Lecture des feuilles Excel

On lit d'abord le classeur avec Pandas afin de pouvoir diagnostiquer les données même si l'import Pandapower échoue.

In [ ]:
feuilles_attendues = ["bus", "line", "switch", "trafo", "ext_grid", "sgen", "load"]
classeur = pd.ExcelFile(chemin_reseau)
print("Feuilles présentes :", classeur.sheet_names)

feuilles_manquantes = sorted(set(feuilles_attendues) - set(classeur.sheet_names))
assert not feuilles_manquantes, f"Feuilles manquantes : {feuilles_manquantes}"

tables = {nom: pd.read_excel(chemin_reseau, sheet_name=nom) for nom in feuilles_attendues}
pd.DataFrame(
    {
        "lignes": {nom: len(table) for nom, table in tables.items()},
        "cellules renseignées": {nom: int(table.notna().sum().sum()) for nom, table in tables.items()},
    }
).loc[feuilles_attendues]

## 3. Inspection des éléments principaux

In [ ]:
display(tables["bus"])
display(tables["line"])
display(tables["trafo"])
display(tables["ext_grid"])
display(tables["load"])

## 4. Contrôles indispensables

Les contrôles portent sur les identifiants de bus, les extrémités des lignes, les paramètres électriques et les valeurs nécessaires à un premier load-flow.

In [ ]:
# Seules les colonnes vertes du classeur sont obligatoires à cette étape.
champs_obligatoires = {
    "bus": ["id", "name", "vn_kv"],
    "line": ["name", "from_bus", "to_bus", "length_km", "r_ohm_per_km",
             "x_ohm_per_km", "c_nf_per_km", "max_i_ka"],
    "trafo": ["name", "hv_bus", "lv_bus", "sn_mva", "vn_hv_kv", "vn_lv_kv",
              "vk_percent", "vkr_percent", "pfe_kw", "i0_percent", "vector_group"],
    "ext_grid": ["name", "bus"],
    "sgen": ["name", "bus", "p_mw"],
    "load": ["name", "bus"],
}

problemes = []
for feuille, colonnes in champs_obligatoires.items():
    table = tables[feuille]
    for colonne in colonnes:
        if colonne not in table.columns:
            problemes.append({"feuille": feuille, "colonne": colonne, "problème": "colonne absente", "lignes": "toutes"})
            continue
        lignes_vides = table.index[table[colonne].isna()].tolist()
        if lignes_vides:
            problemes.append({"feuille": feuille, "colonne": colonne, "problème": "valeur manquante", "lignes": lignes_vides})

ids_bus = set(tables["bus"]["id"].dropna())
for feuille, colonnes_bus in {"line": ["from_bus", "to_bus"], "trafo": ["hv_bus", "lv_bus"],
                              "ext_grid": ["bus"], "load": ["bus"]}.items():
    for colonne in colonnes_bus:
        if colonne in tables[feuille]:
            inconnus = sorted(set(tables[feuille][colonne].dropna()) - ids_bus)
            if inconnus:
                problemes.append({"feuille": feuille, "colonne": colonne, "problème": "bus inconnu", "lignes": inconnus})

rapport = pd.DataFrame(problemes)
if rapport.empty:
    print("✅ Tous les champs indispensables sont renseignés.")
else:
    print(f"❌ {len(rapport)} catégorie(s) de problème détectée(s).")
    display(rapport)

## 5. Import Pandapower et hypothèses provisoires

Cette cellule s'arrête volontairement tant que les colonnes vertes contiennent des problèmes. Les valeurs absentes des colonnes non vertes sont ensuite ajoutées uniquement dans l'objet `net` : le classeur Excel reste inchangé.

In [ ]:
if not rapport.empty:
    raise ValueError("Compléter d'abord les champs signalés dans le classeur Excel.")

from pp_importer import PandaPowerImporter

net = PandaPowerImporter().read_excel(file_path=str(chemin_reseau))

# Hypothèses provisoires pour valider le load-flow, sans modifier Excel.
net.ext_grid.loc[:, "vm_pu"] = 1.0

# 1 kW par bâtiment recensé et cos(phi) = 0.95 inductif.
import numpy as np
nombre_batiments = {0: 5, 1: 9, 2: 2, 3: 3, 4: 4, 5: 3, 6: 6, 7: 4, 8: 1, 9: 3, 10: 1, 11: 1}
net.load.loc[:, "p_mw"] = net.load["bus"].map(nombre_batiments).fillna(0) / 1000
net.load.loc[:, "q_mvar"] = net.load["p_mw"] * np.tan(np.arccos(0.95))

# La carte raccorde 60437 par un câble BT, contrairement aux 20 kV inscrits dans le gabarit.
net.bus.loc[net.bus["name"].eq("60437"), "vn_kv"] = 0.4

# L'énoncé donne une alimentation MT à 18 kV.
net.bus.loc[net.bus["name"].eq("STMT003438HV"), "vn_kv"] = 18.0
net

## 6. Premier load-flow et résultats

Le calcul n'est exécuté qu'après un import réussi. Les limites utilisées ici sont : 0,95–1,05 pu et 100 % de chargement.

In [ ]:
import pandapower as pp

pp.runpp(net)
print("Convergence :", net.converged)
display(net.res_bus)
display(net.res_line)
display(net.res_trafo)

In [ ]:
tensions_hors_limites = net.res_bus.loc[~net.res_bus["vm_pu"].between(0.95, 1.05)]
lignes_surchargees = net.res_line.loc[net.res_line["loading_percent"] > 100]
trafos_surcharges = net.res_trafo.loc[net.res_trafo["loading_percent"] > 100]

print("Bus hors limites :", len(tensions_hors_limites))
display(tensions_hors_limites)
print("Lignes surchargées :", len(lignes_surchargees))
display(lignes_surchargees)
print("Transformateurs surchargés :", len(trafos_surcharges))
display(trafos_surcharges)

## 7. Graphe de la topologie du réseau

En l'absence de coordonnées géographiques dans le classeur, Pandapower génère automatiquement une disposition artificielle du réseau radial. Les charges et les producteurs statiques sont affichés.

In [ ]:
import matplotlib.pyplot as plt
from pandapower.plotting import simple_plot

# simple_plot ne gère pas les valeurs NaN dans le type des producteurs.
if not net.sgen.empty:
    net.sgen["type"] = net.sgen["type"].astype("string").fillna("sgen")
    net.sgen.loc[net.sgen["type"].str.strip().eq(""), "type"] = "sgen"

fig, ax = plt.subplots(figsize=(14, 9))
simple_plot(
    net, ax=ax, show_plot=False,
    plot_loads=True, plot_sgens=True,
    bus_size=1.2, line_width=2.5,
    trafo_size=1.5, ext_grid_size=1.5,
)
ax.set_title("Topologie du réseau électrique de Trey")
plt.show()

## 8. Graphe interactif des résultats du load-flow

Les bus sont colorés selon leur tension en pu et les lignes selon leur taux de chargement. Placez le curseur sur un élément pour afficher ses valeurs.

In [ ]:
from pandapower.plotting.plotly import pf_res_plotly

fig_resultats = pf_res_plotly(
    net,
    auto_open=False,
    climits_volt=(0.95, 1.05),
    climits_load=(0, 100),
    bus_size=14,
    line_width=3,
)
fig_resultats.update_layout(title="Réseau de Trey — tensions et chargements")
fig_resultats.show()